# ✨ บทที่ 6 — Simplify: ทำให้ขอบแปลงสวยงาม

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A-Phrom-A/ssb-workshop/blob/main/06_simplify.ipynb)


> ต่อจาก: [บทที่ 5 — SAM Production](05_sam_production.ipynb)

---

## ทฤษฎี: 3 วิธีในการ Simplify Polygon

| วิธี | หลักการ | ผลลัพธ์ |
|---|---|---|
| `douglas_peucker` | ลดจำนวนจุดตาม tolerance | เร็ว แต่ยังมีมุมแหลม |
| `buffer_clean` | Dilate (+) แล้ว Erode (-) | เนียน แต่ยังเหลี่ยม |
| `smooth` ✅ | buffer_clean + Chaikin + simplify | โค้งมน ดีที่สุด |

### Chaikin Corner-Cutting Algorithm

![6.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/6.png)

---

## ขั้นตอนที่ 1 — Setup

In [ ]:
# เมื่อรัน cell นี้ จะมีหน้าต่างขอ permission ปรากฏขึ้น
# ให้คลิก "Connect to Google Drive" และเลือก Google Account ของคุณ
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/ssb/drive_structure/ch06_simplify'

print("✅ เชื่อมต่อ Google Drive สำเร็จ!")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.1 — Import และ Paths
# ═══════════════════════════════════════════════════════════

import importlib
import os
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


In [ ]:
RASTER_DIR = os.path.join(BASE_DIR, 'data')
SAT_PATH   = os.path.join(RASTER_DIR, 'for sam satelite test.tif')
SCRIPT_DIR = os.path.join(BASE_DIR, 'scripts')


INPUT_GPKG = os.path.join(BASE_DIR, 'data', 'sam_parcels_final.gpkg')
if not os.path.exists(INPUT_GPKG):
    INPUT_GPKG = os.path.join(RASTER_DIR, 'sam overlay.gpkg')
    print(f'⚠️  ใช้ไฟล์ตัวอย่างแทน: {os.path.basename(INPUT_GPKG)}')

OUTPUT_SIMPLIFIED = '/content/output/sam_parcels_simplified.geojson'
os.makedirs(os.path.dirname(OUTPUT_SIMPLIFIED), exist_ok=True)

gdf_raw = gpd.read_file(INPUT_GPKG)
print(f'✅ โหลดแล้ว: {len(gdf_raw):,} Polygon')

---

## ขั้นตอนที่ 2 — ดูปัญหาก่อน Simplify

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.1 — วิเคราะห์จำนวน Vertices ก่อน Simplify
# ═══════════════════════════════════════════════════════════

def count_vertices(geom):
    if geom is None or geom.is_empty: return 0
    if hasattr(geom, 'exterior'):
        return len(geom.exterior.coords) + sum(len(r.coords) for r in geom.interiors)
    if hasattr(geom, 'geoms'): return sum(count_vertices(g) for g in geom.geoms)
    elif hasattr(geom, 'coords'): return len(geom.coords)
    return 0

vertex_counts = gdf_raw.geometry.apply(count_vertices)
print('📊 ก่อน Simplify:')
print(f'   จำนวน Polygon:             {len(gdf_raw):,}')
print(f'   Vertices เฉลี่ยต่อ Polygon:  {vertex_counts.mean():,.0f} จุด')
print(f'   Vertices รวม:               {vertex_counts.sum():,} จุด')
print(f'\n💡 ยิ่งมีจุดมาก → ไฟล์ขนาดใหญ่ + วาดช้า + เส้นเป็นบันได')

---

## ขั้นตอนที่ 3 — โหลด Script และรัน Simplify

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.1 — โหลด simplify.py จาก Drive
# ═══════════════════════════════════════════════════════════

if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

import simplify
importlib.reload(simplify)
from simplify import clean_geojson_boundaries

print('✅ โหลด simplify.py เรียบร้อย')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.2 — รัน Simplify Pipeline
# ═══════════════════════════════════════════════════════════
# ⏱️ ใช้เวลา 1-5 นาที ขึ้นกับจำนวน Polygon

clean_geojson_boundaries(
    input_path=INPUT_GPKG,
    output_path=OUTPUT_SIMPLIFIED,
    tolerance_meters=2.5,       # ความนุ่มของ smooth (เมตร)
    method='smooth',            # buffer + Chaikin + simplify
    smooth_iterations=2,        # Chaikin 2 รอบ
    min_area_sqm=1600,          # ลบแปลง < 1 ไร่
    resolve_overlaps=True,
    verbose_area_tracking=True, # แสดง % พื้นที่แต่ละขั้น
)

print('✅ Simplify เสร็จสิ้น!')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.3 — เปรียบเทียบก่อน-หลัง
# ═══════════════════════════════════════════════════════════


fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, gdf, color, title in [
    (axes[0], gdf_raw, 'red', f'Before: {len(gdf_raw):,} Polygons — Pixelated Edges'),
    (axes[1], gdf_simplified, '#35C778', f'After: {len(gdf_simplified):,} Polygons — Smooth Edges')
]:
    ax.imshow(rgb, extent=extent, origin='upper')
    gdf.to_crs(crs).boundary.plot(ax=ax, color=color, linewidth=0.7)
    ax.set_title(title, fontsize=12); ax.axis('off')

# Zoom area
x_center = (bounds.left + bounds.right) / 2
y_center = (bounds.bottom + bounds.top) / 2
zoom_w = (bounds.right - bounds.left) * 0.05
zoom_h = (bounds.top - bounds.bottom) * 0.05
xmin, xmax = x_center - zoom_w/2, x_center + zoom_w/2
ymin, ymax = y_center - zoom_h/2, y_center + zoom_h/2

for ax, gdf, color in [
    (axes[0], gdf_raw, 'red'),
    (axes[1], gdf_simplified, '#35C778')
]:
    z = inset_axes(ax, width="35%", height="35%", loc="lower left", borderpad=1)
    z.imshow(rgb, extent=extent, origin='upper')
    gdf.to_crs(crs).boundary.plot(ax=z, color=color, linewidth=1.0)
    z.set_xlim(xmin, xmax); z.set_ylim(ymin, ymax); z.set_xticks([]); z.set_yticks([])
    z.set_title('Zoom', fontsize=9)

plt.suptitle('Simplification: Before vs After', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

---


> ➡️ **บทที่ 7:** รวม AI + SAM = แผนที่แปลงปาล์มจริง